# 0825_kimjaehak_017_dongjin_026_threshold_fix

`0826_dongjin_026_saved_ensemble_shap`은 고정 Threshold `0.5`에서 발생한 False Call만 분석했다.
이 후속 실험은 원본 실험을 수정하지 않고, 동일한 `peace_005` 저장 모델에서 고정 `0.5`와 Validation 선택 공통 Threshold를 나란히 재현해 Threshold에 따라 False Call과 SHAP 해석 대상이 어떻게 달라지는지 확인한다.

- `class=0`인데 모델이 `1`로 판정한 경우를 False Call(FP)로 정의한다.
- Test는 Threshold 선택에 사용하지 않는다.
- 여기서 SHAP는 확률 평균 앙상블의 정확한 additive explanation이 아니라, 네 체크포인트 XGBoost 구성원의 raw-margin `mean |SHAP|` 평균이다.


## 1. 설정과 라이브러리

In [1]:
import gc
import hashlib
import platform
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
import xgboost
from IPython.display import display
from sklearn.metrics import average_precision_score

EXPERIMENT_ID = "0825_kimjaehak_017_dongjin_026_threshold_fix"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
FIXED_THRESHOLD = 0.5


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("저장소 루트를 찾지 못했습니다.")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "raw" / "dataset.csv"
MAPPING_PATH = REPO_ROOT / "data" / "raw" / "mapping.json"
MODEL_PATH = REPO_ROOT / "models" / "0825_peace_005_type_expert_fold_ensemble.pkl"

versions = pd.Series(
    {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "shap": shap.__version__,
        "joblib": joblib.__version__,
    },
    name="execution_versions",
)
display(versions)


/Users/hakeee/Desktop/code/siemens_aoi_ML_practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python          3.12.13
numpy             2.5.2
pandas            3.0.5
scikit_learn      1.9.0
xgboost           3.4.1
shap             0.52.0
joblib            1.5.3
Name: execution_versions, dtype: str

## 2. 저장 모델과 원본 데이터 무결성 확인

In [2]:
bundle = joblib.load(MODEL_PATH)

assert bundle["experiment_id"] == "0825_peace_005_type_expert_fold_ensemble"
assert bundle["ensemble_method"] == "equal_probability_mean"
assert bundle.get("deduplicate_rows", False) is False
assert sha256_file(DATA_PATH) == bundle["dataset_sha256"]
assert sha256_file(MAPPING_PATH) == bundle["mapping_sha256"]

GLOBAL_THRESHOLD = float(bundle["global_threshold"])
CHECKPOINTS = list(bundle.get("final_member_checkpoints", bundle["ensemble_checkpoints"]))
INSPECTION_TYPES = sorted(bundle["feature_columns_by_type"])

bundle_summary = pd.Series(
    {
        "schema_version": bundle.get("schema_version"),
        "model": bundle["experiment_id"],
        "checkpoints": CHECKPOINTS,
        "inspection_types": INSPECTION_TYPES,
        "fixed_threshold": FIXED_THRESHOLD,
        "global_threshold": GLOBAL_THRESHOLD,
        "threshold_selected_on": "Validation 70~80%",
        "dataset_sha256_match": True,
        "mapping_sha256_match": True,
    },
    name="bundle_verification",
)
display(bundle_summary)


schema_version                                                  2
model                    0825_peace_005_type_expert_fold_ensemble
checkpoints                                  [0.3, 0.4, 0.5, 0.7]
inspection_types                                  [0, 1, 2, 3, 4]
fixed_threshold                                               0.5
global_threshold                                          0.00071
threshold_selected_on                           Validation 70~80%
dataset_sha256_match                                         True
mapping_sha256_match                                         True
Name: bundle_verification, dtype: object

## 3. 동일 Test 구간과 앙상블 확률 재현

In [3]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
if raw_df.columns[0].startswith("Unnamed:") or raw_df.columns[0] == "":
    raw_df = raw_df.rename(columns={raw_df.columns[0]: RECORD_ID})

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

if "validation_end_time" in bundle:
    validation_end_time = pd.Timestamp(bundle["validation_end_time"])
else:
    timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
    cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
    boundary_position = int(
        np.searchsorted(cumulative_rows, len(raw_df) * 0.80, side="left")
    )
    validation_end_time = timestamp_group_sizes.index[boundary_position]
test_df = raw_df.loc[raw_df[TIME_COLUMN] > validation_end_time].copy()
test_probability = pd.Series(np.nan, index=test_df.index, dtype="float64")

for inspection_type in INSPECTION_TYPES:
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    member_probabilities = []
    for checkpoint in CHECKPOINTS:
        member = bundle["members"][checkpoint][inspection_type]
        feature_columns = member.get(
            "feature_columns", bundle["feature_columns_by_type"][inspection_type]
        )
        X_encoded = member["preprocessor"].transform(type_test[feature_columns])
        member_probabilities.append(member["model"].predict_proba(X_encoded)[:, 1])
        del X_encoded
    test_probability.loc[type_test.index] = np.mean(
        np.vstack(member_probabilities), axis=0
    )

assert test_probability.notna().all()
print("Test rows:", len(test_df))
print("Test positives:", int(test_df[TARGET].sum()))
print("Test period:", test_df[TIME_COLUMN].min(), "~", test_df[TIME_COLUMN].max())


Test rows: 88052
Test positives: 2325
Test period: 1970-10-13 16:54:52+00:00 ~ 1970-11-02 14:21:28+00:00


## 4. Fixed와 공통 Threshold confusion matrix 비교

In [4]:
def evaluate_threshold(y_true, probability, threshold):
    y_true = np.asarray(y_true, dtype="int8")
    prediction = (np.asarray(probability) >= threshold).astype("int8")
    tp = int(((y_true == 1) & (prediction == 1)).sum())
    fn = int(((y_true == 1) & (prediction == 0)).sum())
    fp = int(((y_true == 0) & (prediction == 1)).sum())
    tn = int(((y_true == 0) & (prediction == 0)).sum())
    return {
        "threshold": float(threshold),
        "pr_auc": average_precision_score(y_true, probability),
        "recall": tp / (tp + fn),
        "false_call_reduction": tn / (tn + fp),
        "false_call_rate": fp / (tn + fp),
        "tp": tp,
        "fn": fn,
        "fp": fp,
        "tn": tn,
    }


STRATEGIES = {
    "fixed_0.5": FIXED_THRESHOLD,
    "validation_global": GLOBAL_THRESHOLD,
}
strategy_metrics = pd.DataFrame(
    [
        {"strategy": name, **evaluate_threshold(test_df[TARGET], test_probability, threshold)}
        for name, threshold in STRATEGIES.items()
    ]
).set_index("strategy")

expected_confusion = {
    "fixed_0.5": {"tp": 408, "fn": 1917, "fp": 159, "tn": 85568},
    "validation_global": {"tp": 2184, "fn": 141, "fp": 41118, "tn": 44609},
}
for strategy, expected in expected_confusion.items():
    for metric, expected_value in expected.items():
        actual = int(strategy_metrics.loc[strategy, metric])
        assert actual == expected_value, (strategy, metric, actual, expected_value)

display(strategy_metrics)
print("peace_005 confusion matrix reproduction: PASS")


,threshold,pr_auc,recall,false_call_reduction,false_call_rate,tp,fn,fp,tn
strategy,,,,,,,,,
fixed_0.5,0.50000,0.382545,0.175484,0.998145,0.001855,408,1917,159,85568
validation_global,0.00071,0.382545,0.939355,0.520361,0.479639,2184,141,41118,44609


peace_005 confusion matrix reproduction: PASS


In [5]:
type_confusion_rows = []
for inspection_type in INSPECTION_TYPES:
    type_mask = test_df[TYPE_COLUMN] == inspection_type
    for strategy, threshold in STRATEGIES.items():
        type_confusion_rows.append(
            {
                "inspection_type": inspection_type,
                "strategy": strategy,
                "rows": int(type_mask.sum()),
                **evaluate_threshold(
                    test_df.loc[type_mask, TARGET],
                    test_probability.loc[type_mask],
                    threshold,
                ),
            }
        )

type_confusion = pd.DataFrame(type_confusion_rows).set_index(
    ["strategy", "inspection_type"]
)
display(type_confusion[["rows", "threshold", "recall", "false_call_rate", "tp", "fn", "fp", "tn"]])


,,rows,threshold,recall,false_call_rate,tp,fn,fp,tn
strategy,inspection_type,,,,,,,,
fixed_0.5,0,19491,0.50000,0.000000,0.000000,0,195,0,19296
validation_global,0,19491,0.00071,0.784615,0.429208,153,42,8282,11014
fixed_0.5,1,12351,0.50000,0.348837,0.006046,270,504,70,11507
validation_global,1,12351,0.00071,0.996124,0.679192,771,3,7863,3714
fixed_0.5,2,20543,0.50000,0.083447,0.000353,61,670,7,19805
validation_global,2,20543,0.00071,0.943912,0.558601,690,41,11067,8745
fixed_0.5,3,34939,0.50000,0.125817,0.002389,77,535,82,34245
validation_global,3,34939,0.00071,0.910131,0.384275,557,55,13191,21136
fixed_0.5,4,728,0.50000,0.000000,0.000000,0,13,0,715


## 5. Threshold별 False Call 구성원 평균 SHAP

Threshold로 FP 표본을 먼저 정한 다음, 네 체크포인트 모델 각각의 TreeSHAP를 계산한다. 피처별 `mean(abs(SHAP))`와 signed SHAP 평균을 구하고 네 모델에서 다시 동일 가중 평균한다.

이 값은 구성원 중요도 순위이며, 확률 평균 앙상블 출력의 정확한 additive SHAP로 해석하지 않는다.


In [6]:
def member_mean_shap(inspection_type, frame):
    abs_sum = {}
    signed_sum = {}
    for checkpoint in CHECKPOINTS:
        member = bundle["members"][checkpoint][inspection_type]
        feature_columns = member.get(
            "feature_columns", bundle["feature_columns_by_type"][inspection_type]
        )
        preprocessor = member["preprocessor"]
        X_encoded = preprocessor.transform(frame[feature_columns])
        if hasattr(X_encoded, "toarray"):
            X_encoded = X_encoded.toarray()

        values = shap.TreeExplainer(member["model"]).shap_values(X_encoded)
        if isinstance(values, list):
            values = values[-1]
        values = np.asarray(values)
        names = preprocessor.get_feature_names_out()
        mean_abs = np.abs(values).mean(axis=0)
        mean_signed = values.mean(axis=0)

        for name, abs_value, signed_value in zip(names, mean_abs, mean_signed):
            abs_sum[name] = abs_sum.get(name, 0.0) + float(abs_value)
            signed_sum[name] = signed_sum.get(name, 0.0) + float(signed_value)

        del X_encoded, values
        gc.collect()

    rows = []
    for feature in abs_sum:
        rows.append(
            {
                "feature": feature,
                "member_mean_abs_shap": abs_sum[feature] / len(CHECKPOINTS),
                "member_mean_signed_shap": signed_sum[feature] / len(CHECKPOINTS),
            }
        )
    return pd.DataFrame(rows).sort_values(
        "member_mean_abs_shap", ascending=False, ignore_index=True
    )


shap_rows = []
for strategy, threshold in STRATEGIES.items():
    prediction = test_probability >= threshold
    for inspection_type in INSPECTION_TYPES:
        fp_mask = (
            (test_df[TYPE_COLUMN] == inspection_type)
            & (test_df[TARGET] == 0)
            & prediction
        )
        fp_frame = test_df.loc[fp_mask]
        print(strategy, "type", inspection_type, "FP", len(fp_frame))
        if len(fp_frame) == 0:
            continue
        importance = member_mean_shap(inspection_type, fp_frame).head(10).copy()
        importance.insert(0, "rank", np.arange(1, len(importance) + 1))
        importance.insert(0, "false_calls", len(fp_frame))
        importance.insert(0, "inspection_type", inspection_type)
        importance.insert(0, "strategy", strategy)
        shap_rows.append(importance)

shap_top10 = pd.concat(shap_rows, ignore_index=True)
display(shap_top10)


fixed_0.5 type 0 FP 0
fixed_0.5 type 1 FP 70


fixed_0.5 type 2 FP 7


fixed_0.5 type 3 FP 82


fixed_0.5 type 4 FP 0
validation_global type 0 FP 8282


validation_global type 1 FP 7863


validation_global type 2 FP 11067


validation_global type 3 FP 13191


validation_global type 4 FP 715


,strategy,inspection_type,false_calls,rank,feature,member_mean_abs_shap,member_mean_signed_shap
0,fixed_0.5,1,70,1,continuous__inspection_feat24,1.584373,1.575533
1,fixed_0.5,1,70,2,continuous__inspection_feat48,1.018508,0.137153
2,fixed_0.5,1,70,3,continuous__inspection_feat8,0.759719,0.692814
3,fixed_0.5,1,70,4,categorical__meta_feat4_28,0.742388,0.742388
4,fixed_0.5,1,70,5,categorical__meta_feat1_22,0.475802,0.475499
...,...,...,...,...,...,...,...
75,validation_global,4,715,6,continuous__inspection_feat7,0.000000,0.000000
76,validation_global,4,715,7,continuous__inspection_feat8,0.000000,0.000000
77,validation_global,4,715,8,continuous__inspection_feat9,0.000000,0.000000
78,validation_global,4,715,9,continuous__inspection_feat16,0.000000,0.000000


## 6. Threshold 변경에 따른 상위 피처 비교

In [7]:
comparison_rows = []
for inspection_type in INSPECTION_TYPES:
    fixed = shap_top10.loc[
        (shap_top10["strategy"] == "fixed_0.5")
        & (shap_top10["inspection_type"] == inspection_type)
    ]
    global_policy = shap_top10.loc[
        (shap_top10["strategy"] == "validation_global")
        & (shap_top10["inspection_type"] == inspection_type)
    ]
    fixed_top5 = fixed.head(5)["feature"].tolist()
    global_top5 = global_policy.head(5)["feature"].tolist()
    union = set(fixed_top5) | set(global_top5)
    overlap = set(fixed_top5) & set(global_top5)
    comparison_rows.append(
        {
            "inspection_type": inspection_type,
            "fixed_fp": int(type_confusion.loc[("fixed_0.5", inspection_type), "fp"]),
            "global_fp": int(type_confusion.loc[("validation_global", inspection_type), "fp"]),
            "fixed_top5": fixed_top5,
            "global_top5": global_top5,
            "top5_jaccard": len(overlap) / len(union) if union else np.nan,
        }
    )

threshold_feature_comparison = pd.DataFrame(comparison_rows).set_index("inspection_type")
display(threshold_feature_comparison)

for row in comparison_rows:
    print(
        f"Type {row['inspection_type']} | fixed FP={row['fixed_fp']:,} | "
        f"global FP={row['global_fp']:,} | overlap={row['top5_jaccard']:.3f}"
    )
    print("  fixed top5:", row['fixed_top5'])
    print("  global top5:", row['global_top5'])


,fixed_fp,global_fp,fixed_top5,global_top5,top5_jaccard
inspection_type,,,,,
0,0,8282,[],"[continuous__inspection_feat41, categorical__m...",0.000000
1,70,7863,"[continuous__inspection_feat24, continuous__in...","[continuous__inspection_feat48, continuous__in...",0.666667
2,7,11067,"[continuous__inspection_feat96, categorical__m...","[categorical__meta_feat1_27, continuous__inspe...",0.428571
3,82,13191,"[continuous__inspection_feat22, continuous__in...","[continuous__inspection_feat12, categorical__m...",0.666667
4,0,715,[],"[categorical__meta_feat1_0, continuous__inspec...",0.000000


Type 0 | fixed FP=0 | global FP=8,282 | overlap=0.000
  fixed top5: []
  global top5: ['continuous__inspection_feat41', 'categorical__meta_feat2_1', 'continuous__inspection_feat24', 'categorical__meta_feat1_10', 'categorical__meta_feat4_0']
Type 1 | fixed FP=70 | global FP=7,863 | overlap=0.667
  fixed top5: ['continuous__inspection_feat24', 'continuous__inspection_feat48', 'continuous__inspection_feat8', 'categorical__meta_feat4_28', 'categorical__meta_feat1_22']
  global top5: ['continuous__inspection_feat48', 'continuous__inspection_feat24', 'categorical__meta_feat1_22', 'categorical__meta_feat4_28', 'continuous__inspection_feat4']
Type 2 | fixed FP=7 | global FP=11,067 | overlap=0.429
  fixed top5: ['continuous__inspection_feat96', 'categorical__meta_feat1_27', 'categorical__meta_feat4_3', 'continuous__inspection_feat22', 'categorical__meta_feat4_41']
  global top5: ['categorical__meta_feat1_27', 'continuous__inspection_feat96', 'continuous__inspection_feat12', 'continuous__inspect

## 7. 결론 및 다음 단계

- 동일한 저장 모델과 Test 88,052행에서 `peace_005`의 confusion matrix를 정확히 재현했다.
- 고정 `0.5`는 FP 159건만 SHAP 대상으로 사용하지만, Validation 공통 Threshold `0.0007097449`는 FP 41,118건을 대상으로 사용한다.
- 따라서 `dongjin_026`의 고정 `0.5` SHAP를 Recall 중심 공통 Threshold 정책의 False Call 원인으로 일반화할 수 없다.
- 공통 Threshold에서는 모든 inspection type에 FP가 발생하며, Type 4는 정상 715건 전체가 FP다. Type 4 구성 모델의 SHAP가 모두 0이라면 상수 출력 모델이라는 뜻이므로 별도 정책이 필요하다.
- 이 실험의 SHAP는 FP 표본에서 영향이 컸던 피처의 순위를 보여줄 뿐, 해당 피처가 오판의 원인이라는 인과 해석은 하지 않는다.
- 후속 검증에서는 공통 Threshold의 FP와 TN을 비교한 signed SHAP 차이와 실제 공정 비용을 함께 확인한다.
